In [1]:
import pandas as pd

df = pd.read_csv("../data/customer-purchase-prediction-data.csv")
df

,age,job,marital,education,Credit,balance,housing_loan,personal_loan,contact,last_contact_day,last_contact_month,last_contact_duration/sec,campaign,pdays,previous,previous marketing campaign,subscribed term deposit
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,October,79 seconds,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,May,220 seconds,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,April,185 seconds,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,June,199 seconds,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,May,226 seconds,1,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,no,-333,yes,no,cellular,30,July,329 seconds,5,-1,0,unknown,no
4517,57,self-employed,married,tertiary,yes,-3313,yes,yes,unknown,9,May,153 seconds,1,-1,0,unknown,no
4518,57,technician,married,secondary,no,295,no,no,cellular,19,August,151 seconds,11,-1,0,unknown,no
4519,28,blue-collar,married,secondary,no,1137,no,no,cellular,6,February,129 seconds,4,211,3,other,no


## EDA

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   age                          4521 non-null   int64 
 1   job                          4521 non-null   object
 2   marital                      4521 non-null   object
 3   education                    4521 non-null   object
 4   Credit                       4521 non-null   object
 5   balance                      4521 non-null   int64 
 6   housing_loan                 4521 non-null   object
 7   personal_loan                4521 non-null   object
 8   contact                      4521 non-null   object
 9   last_contact_day             4521 non-null   int64 
 10  last_contact_month           4521 non-null   object
 11  last_contact_duration/sec    4521 non-null   object
 12  campaign                     4521 non-null   int64 
 13  pdays                        4521

In [3]:
df.duplicated().sum()

np.int64(0)

In [4]:
num_cols = df.select_dtypes(include = ['int', 'float']).columns
cat_cols = df.select_dtypes(include = ['object']).columns

print(f'Numerical columns: {len(num_cols)}')
print(f'Categorical columns: {len(cat_cols)}')

Numerical columns: 6
Categorical columns: 11


### Findings:
1. Target column = ```subscribed term deposit```
2. Missing values = 0
3. Duplicates = 0
4. ```last_contact_duration/sec``` column has numeric + text values
5. Some columns have uppercase letters
6. Some columns contain whitespaces, / in their name
7. Dataset has 11 categorical columns

### Pre-processing steps:
1. Encode target column
2. Extract numeric values from ```last_contact_duration/sec``` column
3. Convert all values into lower
4. Remove/replace whitespaces, / from column names
5. Convert all features into numeric values

In [5]:
# 1. Encode target column and other yes/ no value columns

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
yes_no_cols = ['Credit', 'housing_loan', 'personal_loan', 'subscribed term deposit']

for col in yes_no_cols:
    df[col] = le.fit_transform(df[col])

df.head(15)

,age,job,marital,education,Credit,balance,housing_loan,personal_loan,contact,last_contact_day,last_contact_month,last_contact_duration/sec,campaign,pdays,previous,previous marketing campaign,subscribed term deposit
0,30,unemployed,married,primary,0,1787,0,0,cellular,19,October,79 seconds,1,-1,0,unknown,0
1,33,services,married,secondary,0,4789,1,1,cellular,11,May,220 seconds,1,339,4,failure,0
2,35,management,single,tertiary,0,1350,1,0,cellular,16,April,185 seconds,1,330,1,failure,0
3,30,management,married,tertiary,0,1476,1,1,unknown,3,June,199 seconds,4,-1,0,unknown,0
4,59,blue-collar,married,secondary,0,0,1,0,unknown,5,May,226 seconds,1,-1,0,unknown,0
5,35,management,single,tertiary,0,747,0,0,cellular,23,February,141 seconds,2,176,3,failure,0
6,36,self-employed,married,tertiary,0,307,1,0,cellular,14,May,341 seconds,1,330,2,other,0
7,39,technician,married,secondary,0,147,1,0,cellular,6,May,151 seconds,2,-1,0,unknown,0
8,41,entrepreneur,married,tertiary,0,221,1,0,unknown,14,May,57 seconds,2,-1,0,unknown,0
9,43,services,married,primary,0,-88,1,1,cellular,17,April,313 seconds,1,147,2,failure,0


In [6]:
# 2. Fix on last_contact_duration/sec column

df['last_contact_duration/sec'] = df['last_contact_duration/sec'].str.split(' ').str[0].astype(int)
df

,age,job,marital,education,Credit,balance,housing_loan,personal_loan,contact,last_contact_day,last_contact_month,last_contact_duration/sec,campaign,pdays,previous,previous marketing campaign,subscribed term deposit
0,30,unemployed,married,primary,0,1787,0,0,cellular,19,October,79,1,-1,0,unknown,0
1,33,services,married,secondary,0,4789,1,1,cellular,11,May,220,1,339,4,failure,0
2,35,management,single,tertiary,0,1350,1,0,cellular,16,April,185,1,330,1,failure,0
3,30,management,married,tertiary,0,1476,1,1,unknown,3,June,199,4,-1,0,unknown,0
4,59,blue-collar,married,secondary,0,0,1,0,unknown,5,May,226,1,-1,0,unknown,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,0,-333,1,0,cellular,30,July,329,5,-1,0,unknown,0
4517,57,self-employed,married,tertiary,1,-3313,1,1,unknown,9,May,153,1,-1,0,unknown,0
4518,57,technician,married,secondary,0,295,0,0,cellular,19,August,151,11,-1,0,unknown,0
4519,28,blue-collar,married,secondary,0,1137,0,0,cellular,6,February,129,4,211,3,other,0


In [7]:
# 3. Fix column names by replacing spaces and slashes with underscores

df.columns = df.columns.str.replace(r'[ /]', '_', regex=True)

df

,age,job,marital,education,Credit,balance,housing_loan,personal_loan,contact,last_contact_day,last_contact_month,last_contact_duration_sec,campaign,pdays,previous,previous_marketing_campaign,subscribed_term_deposit
0,30,unemployed,married,primary,0,1787,0,0,cellular,19,October,79,1,-1,0,unknown,0
1,33,services,married,secondary,0,4789,1,1,cellular,11,May,220,1,339,4,failure,0
2,35,management,single,tertiary,0,1350,1,0,cellular,16,April,185,1,330,1,failure,0
3,30,management,married,tertiary,0,1476,1,1,unknown,3,June,199,4,-1,0,unknown,0
4,59,blue-collar,married,secondary,0,0,1,0,unknown,5,May,226,1,-1,0,unknown,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,services,married,secondary,0,-333,1,0,cellular,30,July,329,5,-1,0,unknown,0
4517,57,self-employed,married,tertiary,1,-3313,1,1,unknown,9,May,153,1,-1,0,unknown,0
4518,57,technician,married,secondary,0,295,0,0,cellular,19,August,151,11,-1,0,unknown,0
4519,28,blue-collar,married,secondary,0,1137,0,0,cellular,6,February,129,4,211,3,other,0


In [8]:
# 4. Convert values to lowercase

cat_cols = df.select_dtypes(include = ['object']).columns # since we changed the last_contact_duration/sec into int and renamed some columns

for col in cat_cols:
    df[col] = df[col].str.lower()

df.head(15)

,age,job,marital,education,Credit,balance,housing_loan,personal_loan,contact,last_contact_day,last_contact_month,last_contact_duration_sec,campaign,pdays,previous,previous_marketing_campaign,subscribed_term_deposit
0,30,unemployed,married,primary,0,1787,0,0,cellular,19,october,79,1,-1,0,unknown,0
1,33,services,married,secondary,0,4789,1,1,cellular,11,may,220,1,339,4,failure,0
2,35,management,single,tertiary,0,1350,1,0,cellular,16,april,185,1,330,1,failure,0
3,30,management,married,tertiary,0,1476,1,1,unknown,3,june,199,4,-1,0,unknown,0
4,59,blue-collar,married,secondary,0,0,1,0,unknown,5,may,226,1,-1,0,unknown,0
5,35,management,single,tertiary,0,747,0,0,cellular,23,february,141,2,176,3,failure,0
6,36,self-employed,married,tertiary,0,307,1,0,cellular,14,may,341,1,330,2,other,0
7,39,technician,married,secondary,0,147,1,0,cellular,6,may,151,2,-1,0,unknown,0
8,41,entrepreneur,married,tertiary,0,221,1,0,unknown,14,may,57,2,-1,0,unknown,0
9,43,services,married,primary,0,-88,1,1,cellular,17,april,313,1,147,2,failure,0


In [9]:
# 5. Encoding other cateorical columns

for col in cat_cols:
    print(f'{df[col].name}:{df[col].unique()}\n')

job:['unemployed' 'services' 'management' 'blue-collar' 'self-employed'
 'technician' 'entrepreneur' 'admin.' 'student' 'housemaid' 'retired'
 'unknown']

marital:['married' 'single' 'divorced']

education:['primary' 'secondary' 'tertiary' 'unknown']

contact:['cellular' 'unknown' 'telephone']

last_contact_month:['october' 'may' 'april' 'june' 'february' 'august' 'january' 'july'
 'november' 'september' 'march' 'december']

previous_marketing_campaign:['unknown' 'failure' 'other' 'success']



In [10]:
one_hot_cols = ['job', 'marital', 'contact','previous_marketing_campaign']

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output = False, drop = 'first') # drop first to avoid dummy variable trap
for col in one_hot_cols:
    encoded = encoder.fit_transform(df[[col]])
    encoded_df = pd.DataFrame(encoded, columns = encoder.get_feature_names_out([col]))
    df = pd.concat([df, encoded_df], axis = 1)
    df.drop(col, axis = 1, inplace =True)

df.head(15)

,age,education,Credit,balance,housing_loan,personal_loan,last_contact_day,last_contact_month,last_contact_duration_sec,campaign,...,job_technician,job_unemployed,job_unknown,marital_married,marital_single,contact_telephone,contact_unknown,previous_marketing_campaign_other,previous_marketing_campaign_success,previous_marketing_campaign_unknown
0,30,primary,0,1787,0,0,19,october,79,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,33,secondary,0,4789,1,1,11,may,220,1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,35,tertiary,0,1350,1,0,16,april,185,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,30,tertiary,0,1476,1,1,3,june,199,4,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,59,secondary,0,0,1,0,5,may,226,1,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
5,35,tertiary,0,747,0,0,23,february,141,2,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
6,36,tertiary,0,307,1,0,14,may,341,1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
7,39,secondary,0,147,1,0,6,may,151,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
8,41,tertiary,0,221,1,0,14,may,57,2,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
9,43,primary,0,-88,1,1,17,april,313,1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 31 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   age                                  4521 non-null   int64  
 1   education                            4521 non-null   object 
 2   Credit                               4521 non-null   int64  
 3   balance                              4521 non-null   int64  
 4   housing_loan                         4521 non-null   int64  
 5   personal_loan                        4521 non-null   int64  
 6   last_contact_day                     4521 non-null   int64  
 7   last_contact_month                   4521 non-null   object 
 8   last_contact_duration_sec            4521 non-null   int64  
 9   campaign                             4521 non-null   int64  
 10  pdays                                4521 non-null   int64  
 11  previous                      

In [12]:
# Map education column to numerical values

df['education'] = df['education'].map({'primary': 1, 'secondary': 2, 'tertiary': 3, 'unknown': 0})
df['education'].unique()

array([1, 2, 3, 0])

In [13]:
# Map last_contact_month column to numerical values

df['last_contact_month'] = df['last_contact_month'].map({'january': 1, 'february': 2, 'march': 3, 'april': 4, 'may': 5, 'june': 6, 'july': 7, 'august': 8, 'september': 9, 'october': 10, 'november': 11, 'december': 12})
df['last_contact_month'].unique()

array([10,  5,  4,  6,  2,  8,  1,  7, 11,  9,  3, 12])

In [14]:
df

,age,education,Credit,balance,housing_loan,personal_loan,last_contact_day,last_contact_month,last_contact_duration_sec,campaign,...,job_technician,job_unemployed,job_unknown,marital_married,marital_single,contact_telephone,contact_unknown,previous_marketing_campaign_other,previous_marketing_campaign_success,previous_marketing_campaign_unknown
0,30,1,0,1787,0,0,19,10,79,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,33,2,0,4789,1,1,11,5,220,1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,35,3,0,1350,1,0,16,4,185,1,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,30,3,0,1476,1,1,3,6,199,4,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,59,2,0,0,1,0,5,5,226,1,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4516,33,2,0,-333,1,0,30,7,329,5,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4517,57,3,1,-3313,1,1,9,5,153,1,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4518,57,2,0,295,0,0,19,8,151,11,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4519,28,2,0,1137,0,0,6,2,129,4,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 31 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   age                                  4521 non-null   int64  
 1   education                            4521 non-null   int64  
 2   Credit                               4521 non-null   int64  
 3   balance                              4521 non-null   int64  
 4   housing_loan                         4521 non-null   int64  
 5   personal_loan                        4521 non-null   int64  
 6   last_contact_day                     4521 non-null   int64  
 7   last_contact_month                   4521 non-null   int64  
 8   last_contact_duration_sec            4521 non-null   int64  
 9   campaign                             4521 non-null   int64  
 10  pdays                                4521 non-null   int64  
 11  previous                      